# SegFormer - Efficient Semantic Segmentation with Transformers

**SegFormer** uses a hierarchical transformer encoder (MiT) + lightweight MLP decoder. More efficient than flat ViT-based SETR, often with better results.

Proposed by NVIDIA (NeurIPS 2021).  
Paper: [SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers](https://arxiv.org/abs/2105.15203)

**Architecture:**
- **Encoder**: MiT (Mix Transformer) - hierarchical, no positional encodings
- **Decoder**: Simple MLP that fuses multi-scale features

**Dependencies:** PyTorch + torchvision only. SegFormer (MiT encoder + MLP decoder) is defined in `segformer_torch.py` alongside this notebook — no Hugging Face `transformers` install.

If imports fail, set the notebook kernel working directory to this folder (`Segmentation_Transformers`), or run from the repo root — the next cell adds common paths automatically.

In [1]:
import os
import sys
from pathlib import Path
_cwd = Path.cwd().resolve()
for _d in [
    _cwd,
    _cwd / "Segmentation_Transformers",
    _cwd / "BRISC" / "Segmentation_Transformers",
]:
    if (_d / "segformer_torch.py").is_file():
        sys.path.insert(0, str(_d))
        break

import math
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from segformer_torch import SegFormer

# Reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [2]:
# ============== CONFIG ==============
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 40
LR = 1e-4
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.1
MIN_LR_RATIO = 0.01
MAX_GRAD_NORM = 1.0
NUM_CLASSES = 1
# BCE + Dice combined loss (tunable)
DICE_LAMBDA = 0.5

# SegFormer variant: 'b0', 'b1', or 'b2' (see segformer_torch.py). b0 is lightest.
MODEL_SIZE = 'b0'
BASE_PATH = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task"
TRAIN_IMG = os.path.join(BASE_PATH, "train", "images")
TRAIN_MASK = os.path.join(BASE_PATH, "train", "masks")
TEST_IMG = os.path.join(BASE_PATH, "test", "images")
TEST_MASK = os.path.join(BASE_PATH, "test", "masks")

PRED_BASE = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/SegFormer"
PRED_MASK_DIR = os.path.join(PRED_BASE, "masks")
OVERLAY_DIR = os.path.join(PRED_BASE, "overlays")
MODEL_SAVE_DIR = os.path.join(os.path.dirname(PRED_BASE), "models")
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, f"segformer_{MODEL_SIZE}.pth")
BEST_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, f"segformer_{MODEL_SIZE}_best.pth")

os.makedirs(PRED_MASK_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


## Training improvements (Phase 1)

- **Augmentation (train only):** paired H/V flip + `ColorJitter` on images (mask follows geometry only).
- **Optimizer:** **AdamW** + weight decay.
- **Schedule:** **linear warmup** then **cosine decay** to `MIN_LR_RATIO × LR` (total steps = epochs × steps/epoch).
- **Stability:** gradient clipping (`MAX_GRAD_NORM`).
- **Epochs:** 40 by default (raise if val loss still dropping).

Same BRISC paths as `DeepLabV3plus/DeepLabV3ii.ipynb`. Encoder still trains **from scratch**; for biggest gains add pretrained weights (HF SegFormer) later.


## 1. Dataset & Preprocessing

ImageNet **normalize** for inputs. Train uses **augmentation**; val/test does not.

In [3]:
IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]


class SegDataset(Dataset):
    """Paired image/mask; optional train-time augmentation (geometry on both, color on image only)."""

    def __init__(self, img_dir, mask_dir, augment=False):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.augment = augment
        self.normalize = transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)
        self.color_jitter = transforms.ColorJitter(
            brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        image = image.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        mask = mask.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)

        if self.augment:
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)
                mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
            if random.random() < 0.9:
                image = self.color_jitter(image)

        image = transforms.ToTensor()(image)
        image = self.normalize(image)
        mask = np.array(mask).astype("float32") / 255.0
        mask = (mask > 0.5).astype("float32")
        mask = torch.tensor(mask)

        out_name = self.masks[idx]
        return image, mask, out_name


def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    masks = torch.stack([b[1] for b in batch])
    names = [b[2] for b in batch]
    return images, masks, names

## 2. Model Setup

`SegFormer(variant=MODEL_SIZE)` with `num_classes=1`; BCE + Dice in the training loop below.

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SegFormer(num_classes=NUM_CLASSES, variant=MODEL_SIZE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(f"Device: {device}")
print(f"Backbone: {MODEL_SIZE}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")


Device: cuda
Backbone: b0
Params: 3,716,705


In [5]:
train_dataset = SegDataset(TRAIN_IMG, TRAIN_MASK, augment=True)
test_dataset = SegDataset(TEST_IMG, TEST_MASK, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

## 3. Training loop

LR: linear **warmup** then **cosine** decay. SegFormer logits are interpolated to mask size for the loss.

In [6]:
steps_per_epoch = len(train_loader)
total_steps = max(1, EPOCHS * steps_per_epoch)
warmup_steps = max(1, int(WARMUP_RATIO * total_steps))


def _lr_lambda(step):
    # step is 0 on first batch; use (step+1)/warmup so LR is non-zero immediately
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

bce_loss = nn.BCEWithLogitsLoss()


def dice_loss_logits(logits, targets, smooth=1e-6):
    p = torch.sigmoid(logits).view(logits.shape[0], -1)
    t = targets.view(targets.shape[0], -1)
    inter = (p * t).sum(dim=1)
    union = p.sum(dim=1) + t.sum(dim=1)
    dice = (2 * inter + smooth) / (union + smooth)
    return (1 - dice).mean()


def seg_loss(logits, masks):
    return bce_loss(logits, masks) + DICE_LAMBDA * dice_loss_logits(logits, masks)


def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for images, masks, _ in loader:
        images = images.to(device)
        masks = masks.unsqueeze(1).to(device)
        optimizer.zero_grad()
        logits = model(images)
        logits = F.interpolate(logits, size=masks.shape[2:], mode='bilinear', align_corners=False)
        loss = seg_loss(logits, masks)
        loss.backward()
        if MAX_GRAD_NORM > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks = masks.unsqueeze(1).to(device)
            logits = model(images)
            logits = F.interpolate(logits, size=masks.shape[2:], mode='bilinear', align_corners=False)
            loss = seg_loss(logits, masks)
            total_loss += loss.item()
    return total_loss / len(loader)


In [7]:
best_val_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss = validate(model, test_loader, device)
    lr_now = optimizer.param_groups[0]["lr"]
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    print(
        f"Epoch [{epoch}/{EPOCHS}] Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} LR: {lr_now:.2e}"
    )


Epoch [1/40] Loss: 1.0739 Val Loss: 0.8098 LR: 2.51e-05
Epoch [2/40] Loss: 0.6300 Val Loss: 0.6267 LR: 5.01e-05
Epoch [3/40] Loss: 0.4782 Val Loss: 0.3933 LR: 7.51e-05
Epoch [4/40] Loss: 0.3561 Val Loss: 0.3020 LR: 1.00e-04
Epoch [5/40] Loss: 0.2704 Val Loss: 0.2337 LR: 9.98e-05
Epoch [6/40] Loss: 0.2278 Val Loss: 0.2130 LR: 9.92e-05
Epoch [7/40] Loss: 0.2048 Val Loss: 0.2002 LR: 9.83e-05
Epoch [8/40] Loss: 0.1879 Val Loss: 0.1876 LR: 9.70e-05
Epoch [9/40] Loss: 0.1778 Val Loss: 0.1802 LR: 9.54e-05
Epoch [10/40] Loss: 0.1657 Val Loss: 0.1659 LR: 9.34e-05
Epoch [11/40] Loss: 0.1598 Val Loss: 0.1564 LR: 9.10e-05
Epoch [12/40] Loss: 0.1503 Val Loss: 0.1518 LR: 8.84e-05
Epoch [13/40] Loss: 0.1462 Val Loss: 0.1460 LR: 8.55e-05
Epoch [14/40] Loss: 0.1417 Val Loss: 0.1403 LR: 8.23e-05
Epoch [15/40] Loss: 0.1372 Val Loss: 0.1435 LR: 7.89e-05
Epoch [16/40] Loss: 0.1334 Val Loss: 0.1336 LR: 7.52e-05
Epoch [17/40] Loss: 0.1283 Val Loss: 0.1363 LR: 7.14e-05
Epoch [18/40] Loss: 0.1236 Val Loss: 0.1

In [8]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")
print(f"Best val checkpoint: {BEST_MODEL_PATH}")


Model saved to /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/models/segformer_b0.pth
Best val checkpoint: /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/models/segformer_b0_best.pth


## 4. Inference & Visualization

In [9]:
def denormalize(tensor):
    """Denormalize for overlay visualization (match tensor device/dtype)."""
    mean = torch.tensor(IMAGE_MEAN, device=tensor.device, dtype=tensor.dtype).view(1, 3, 1, 1)
    std = torch.tensor(IMAGE_STD, device=tensor.device, dtype=tensor.dtype).view(1, 3, 1, 1)
    return tensor * std + mean


model.eval()
torch.set_grad_enabled(False)

with torch.no_grad():
    for images, masks, names in test_loader:
        images = images.to(device)
        logits = model(images)
        logits = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        preds = torch.sigmoid(logits).cpu().numpy()
        preds_bin = (preds > 0.5).astype(np.uint8)

        for i, name in enumerate(names):
            pred_mask = preds_bin[i, 0]
            mask_out = Image.fromarray((pred_mask * 255).astype(np.uint8))
            mask_out.save(os.path.join(PRED_MASK_DIR, name))

            image_vis = denormalize(images[i : i + 1]).squeeze(0).permute(1, 2, 0).cpu().numpy()
            image_vis = np.clip(image_vis, 0, 1)
            overlay = image_vis.copy()
            overlay[pred_mask > 0] = [1, 0, 0]
            combined = 0.7 * image_vis + 0.3 * overlay
            combined = (np.clip(combined, 0, 1) * 255).astype(np.uint8)
            Image.fromarray(combined).save(os.path.join(OVERLAY_DIR, name))

print(f"Saved {len(test_dataset)} masks and overlays (filenames match ground truth)")


Saved 860 masks and overlays (filenames match ground truth)


## 5. Test metrics & comparison report

Run **inference** above first so `masks/` and `overlays/` exist. If you skipped training, run **load checkpoint** then **re-run the inference cell**, then metrics and PDF.


In [10]:
# Load saved .pth before inference if you skipped training (after model + DataLoader cells).
# Uses best validation weights (same pattern as DeepLabV3+).
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()


SegFormer(
  (encoder): MixTransformerEncoder(
    (patch_embeds): ModuleList(
      (0): OverlapPatchEmbed(
        (proj): Conv2d(3, 32, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
        (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      )
      (1): OverlapPatchEmbed(
        (proj): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      )
      (2): OverlapPatchEmbed(
        (proj): Conv2d(64, 160, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
      )
      (3): OverlapPatchEmbed(
        (proj): Conv2d(160, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
    )
    (blocks): ModuleList(
      (0): ModuleList(
        (0-1): 2 x MixTransformerBlock(
          (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=

In [11]:
# Run after inference so PRED_MASK_DIR is populated.
def dice_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)

def iou_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

def precision_recall_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    return precision, recall

def accuracy_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    return np.sum(y_true == y_pred) / len(y_true)

dice_scores, iou_scores, precision_scores, recall_scores, accuracy_scores = [], [], [], [], []
for i in range(len(test_dataset)):
    _, mask, name = test_dataset[i]
    gt = (mask.numpy() > 0.5).astype(np.uint8)
    pred_path = os.path.join(PRED_MASK_DIR, name)
    pred_img = np.array(Image.open(pred_path))
    pred = (pred_img > 127).astype(np.uint8)
    dice_scores.append(dice_np(gt, pred))
    iou_scores.append(iou_np(gt, pred))
    p, r = precision_recall_np(gt, pred)
    precision_scores.append(p)
    recall_scores.append(r)
    accuracy_scores.append(accuracy_np(gt, pred))

print("===== TEST SET RESULTS =====")
print(f"Mean Dice     : {np.mean(dice_scores):.4f}")
print(f"Mean IoU      : {np.mean(iou_scores):.4f}")
print(f"Mean Precision: {np.mean(precision_scores):.4f}")
print(f"Mean Recall   : {np.mean(recall_scores):.4f}")
print(f"Mean Accuracy : {np.mean(accuracy_scores):.4f}")


===== TEST SET RESULTS =====
Mean Dice     : 0.8273
Mean IoU      : 0.7439
Mean Precision: 0.8370
Mean Recall   : 0.8533
Mean Accuracy : 0.9951


In [12]:
# Ground truth vs predicted comparison PDF (Image, GT, Predicted, Overlay, Comparison)
from matplotlib.backends.backend_pdf import PdfPages

SAMPLES_PER_PAGE = 4
COMPARISON_PDF_PATH = os.path.join(PRED_BASE, "comparison_report.pdf")


def make_comparison_overlay(gt_mask, pred_mask):
    """TP=green, FP=red, FN=blue."""
    gt = (gt_mask > 0.5).astype(np.uint8)
    pred = (pred_mask > 0.5).astype(np.uint8)
    h, w = gt.shape
    overlay = np.zeros((h, w, 3), dtype=np.uint8)
    overlay[gt == 1] = [0, 0, 128]
    overlay[pred == 1] = [128, 0, 0]
    overlay[(gt == 1) & (pred == 1)] = [0, 128, 0]
    return overlay


indices = np.arange(len(test_dataset))

with PdfPages(COMPARISON_PDF_PATH) as pdf:
    for page_start in range(0, len(indices), SAMPLES_PER_PAGE):
        page_indices = indices[page_start : page_start + SAMPLES_PER_PAGE]
        n_rows = len(page_indices)
        fig, axes = plt.subplots(n_rows, 5, figsize=(15, 3 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        for row, idx in enumerate(page_indices):
            img_name = test_dataset.images[idx]
            mask_name = test_dataset.masks[idx]
            img_path = os.path.join(TEST_IMG, img_name)
            gt_path = os.path.join(TEST_MASK, mask_name)
            pred_path = os.path.join(PRED_MASK_DIR, mask_name)
            overlay_path = os.path.join(OVERLAY_DIR, mask_name)

            image = np.array(Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            gt_mask = np.array(Image.open(gt_path).convert("L").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            pred_mask = np.array(Image.open(pred_path).convert("L")) / 255.0
            if pred_mask.shape != (IMG_SIZE, IMG_SIZE):
                pred_mask = np.array(Image.fromarray((pred_mask * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0

            overlay_img = np.array(Image.open(overlay_path).convert("RGB")) / 255.0
            if overlay_img.shape[:2] != (IMG_SIZE, IMG_SIZE):
                overlay_img = np.array(Image.fromarray((overlay_img * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0

            comparison = make_comparison_overlay(gt_mask, pred_mask)

            axes[row, 0].imshow(image)
            axes[row, 0].set_title("Image")
            axes[row, 0].axis("off")

            axes[row, 1].imshow(gt_mask, cmap="gray")
            axes[row, 1].set_title("Ground Truth")
            axes[row, 1].axis("off")

            axes[row, 2].imshow(pred_mask, cmap="gray")
            axes[row, 2].set_title("Predicted")
            axes[row, 2].axis("off")

            axes[row, 3].imshow(overlay_img)
            axes[row, 3].set_title("Overlay")
            axes[row, 3].axis("off")

            axes[row, 4].imshow(comparison)
            axes[row, 4].set_title("Comparison (TP=green, FP=red, FN=blue)")
            axes[row, 4].axis("off")

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Comparison PDF saved to {COMPARISON_PDF_PATH}")


Comparison PDF saved to /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/SegFormer/comparison_report.pdf
